## Requesting and Reading Data

In [ ]:
"""
Pull daily precipitation totals for all NCEI GHCN-Daily stations in New York City,
across a continuous date range (used for full-network flood-plausible-day matching).

Requires a free NCEI CDO API token: https://www.ncei.noaa.gov/cdo-web/token
"""

import requests
import time
import csv
import pandas as pd

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------

# CDO token is read from a gitignored file so it never lands in version control.
# Setup: copy .ncei_token.example to .ncei_token and paste your token as the only line.
with open(".ncei_token") as _f:
    TOKEN = _f.read().strip()
HEADERS = {"token": TOKEN}
BASE_URL = "https://www.ncei.noaa.gov/cdo-web/api/v2"

# Bounding box covering all five boroughs (lat/lon, roughly)
# format required by CDO API: minlat, minlon, maxlat, maxlon
NYC_EXTENT = "40.49,-74.27,40.92,-73.68"

# Continuous pull range -- matches the 4_sensor_activity analysis-window cutoff
# (latest date in 2_tidal_analysis/tidal_unified.geojson).
START_DATE = "2025-01-01"
END_DATE = "2026-07-02"

REQUEST_DELAY = 0.25  # keep well under 5 requests/second

In [2]:
# ---------------------------------------------------------------------------
# STEP 1: Find every GHCND station within the NYC bounding box
# ---------------------------------------------------------------------------

def get_nyc_stations():
    stations = []
    offset = 1
    limit = 1000
    while True:
        params = {
            "datasetid": "GHCND",
            "extent": NYC_EXTENT,
            "limit": limit,
            "offset": offset,
        }
        resp = requests.get(f"{BASE_URL}/stations", headers=HEADERS, params=params)
        resp.raise_for_status()
        payload = resp.json()
        results = payload.get("results", [])
        stations.extend(results)

        total = payload.get("metadata", {}).get("resultset", {}).get("count", 0)
        if offset + limit > total or not results:
            break
        offset += limit
        time.sleep(REQUEST_DELAY)

    return stations

In [ ]:
# ---------------------------------------------------------------------------
# STEP 2: Pull PRCP data for one station across a date range, then filter
# ---------------------------------------------------------------------------

def get_station_precip(station_id, start_date=START_DATE, end_date=END_DATE):
    # records[date_str] = {"PRCP": value, "SNOW": value, "TMIN": value}
    records = {}
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)

    # CDO's /data endpoint is queried per calendar year (mirrors the API's own
    # annual convention) so a multi-year range like 2025-01-01..2026-07-02 works.
    for year in range(start.year, end.year + 1):
        year_start = max(start, pd.Timestamp(f"{year}-01-01"))
        year_end = min(end, pd.Timestamp(f"{year}-12-31"))

        offset = 1
        limit = 1000
        while True:
            params = {
                "datasetid": "GHCND",
                "datatypeid": "PRCP,SNOW,TMIN",
                "stationid": station_id,
                "startdate": year_start.strftime("%Y-%m-%d"),
                "enddate": year_end.strftime("%Y-%m-%d"),
                "units": "metric",
                "limit": limit,
                "offset": offset,
            }
            resp = requests.get(f"{BASE_URL}/data", headers=HEADERS, params=params)
            if resp.status_code != 200:
                break
            payload = resp.json()
            results = payload.get("results", [])
            for r in results:
                date_str = r["date"][:10]
                records.setdefault(date_str, {})[r["datatype"]] = r["value"]

            total = payload.get("metadata", {}).get("resultset", {}).get("count", 0)
            if offset + limit > total or not results:
                break
            offset += limit
            time.sleep(REQUEST_DELAY)

    return records

In [4]:
# ---------------------------------------------------------------------------
# STEP 3: Classify each day's precipitation as rain, snow, or none
# ---------------------------------------------------------------------------

def classify_precip(prcp_mm, snow_mm, tmin_c):
    if prcp_mm is None or prcp_mm <= 0:
        return "none"
    if snow_mm is not None and snow_mm > 0:
        return "snow"
    if tmin_c is not None and tmin_c <= 0:
        return "likely snow"  # precip present, below freezing, but no measured snow depth
    return "rain"

In [ ]:
import os

def main(force_refresh=False, start_date=START_DATE, end_date=END_DATE):
    out_path = "nyc_precipitation_by_date.csv"

    if os.path.exists(out_path) and not force_refresh:
        print(f"Found existing {out_path}, skipping API pull.")
        df = pd.read_csv(out_path, parse_dates=["date"])
        return df

    print("Fetching NYC station list...")
    stations = get_nyc_stations()
    print(f"Found {len(stations)} stations in the NYC bounding box.")

    all_dates = pd.date_range(start_date, end_date, freq="D").strftime("%Y-%m-%d")

    rows = []
    for i, station in enumerate(stations, 1):
        station_id = station["id"]
        name = station.get("name", "")
        lat = station.get("latitude", "")
        lon = station.get("longitude", "")

        print(f"[{i}/{len(stations)}] Pulling precipitation for {station_id} ({name})...")
        precip_by_date = get_station_precip(station_id, start_date, end_date)

        for date_str in all_dates:
            day_data = precip_by_date.get(date_str)
            if day_data is None or day_data.get("PRCP") is None:
                continue

            prcp_mm = day_data["PRCP"] / 10.0
            snow_mm = day_data.get("SNOW")
            tmin_c = day_data.get("TMIN")
            tmin_c = tmin_c / 10.0 if tmin_c is not None else None

            precip_type = classify_precip(prcp_mm, snow_mm, tmin_c)

            rows.append({
                "station_id": station_id,
                "station_name": name,
                "latitude": lat,
                "longitude": lon,
                "date": date_str,
                "precipitation_mm": prcp_mm,
                "precipitation_in": round(prcp_mm / 25.4, 3),
                "snow_mm": snow_mm,
                "tmin_c": tmin_c,
                "precip_type": precip_type,
            })

        time.sleep(REQUEST_DELAY)

    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)
    print(f"Fetched and cached {len(df)} rows across {len(stations)} stations "
          f"({start_date} to {end_date})")
    return df

In [ ]:
df = main()  # loads the cached continuous CSV if present, otherwise pulls fresh from NCEI
df.head()

## Understanding the Data

In [6]:
import pandas as pd
df = pd.read_csv("nyc_precipitation_by_date.csv")
rain_dates = df.loc[df["precip_type"] == "rain", "date"].unique()
print(f"{len(rain_dates)} dates had rain")
display(sorted(rain_dates))

83 dates had rain


['2025-01-18',
 '2025-01-19',
 '2025-01-20',
 '2025-01-31',
 '2025-02-08',
 '2025-02-09',
 '2025-02-11',
 '2025-02-12',
 '2025-02-13',
 '2025-02-16',
 '2025-03-05',
 '2025-03-06',
 '2025-03-16',
 '2025-03-18',
 '2025-03-21',
 '2025-03-24',
 '2025-03-25',
 '2025-03-26',
 '2025-03-27',
 '2025-03-29',
 '2025-03-30',
 '2025-03-31',
 '2025-04-02',
 '2025-04-03',
 '2025-04-04',
 '2025-04-05',
 '2025-04-07',
 '2025-04-09',
 '2025-04-13',
 '2025-04-16',
 '2025-04-21',
 '2025-04-22',
 '2025-04-25',
 '2025-04-26',
 '2025-04-27',
 '2025-05-03',
 '2025-05-04',
 '2025-05-06',
 '2025-05-07',
 '2025-05-08',
 '2025-05-10',
 '2025-05-13',
 '2025-05-17',
 '2025-05-21',
 '2025-05-30',
 '2025-06-07',
 '2025-06-09',
 '2025-06-10',
 '2025-06-11',
 '2025-06-12',
 '2025-06-23',
 '2025-06-25',
 '2025-07-16',
 '2025-07-25',
 '2025-07-26',
 '2025-07-31',
 '2025-08-13',
 '2025-08-14',
 '2025-08-17',
 '2025-08-20',
 '2025-09-05',
 '2025-09-18',
 '2025-10-08',
 '2025-10-30',
 '2025-11-03',
 '2025-11-08',
 '2025-11-

In [ ]:
raw = pd.read_csv("../FloodNet_Data_Manual_QC.csv")  # columns: Date, Count, rain(optional), Locations
raw["date"] = pd.to_datetime(raw["Date"])

# Split the comma-separated location list into one row per sensor
raw["Sensors"] = raw["Sensors"].str.split(",")
flattened = raw.explode("Sensors")
flattened["Sensors"] = flattened["Sensors"].str.strip()

triggers = flattened[["date", "Sensors"]].rename(columns={"Sensors": "sensor_name"})
print(triggers.head(10))

Bring in location data

In [ ]:
# geospatial file: assume columns like sensor_id, location_name, latitude, longitude
sensors_geo = pd.read_csv("../../final_deployed_sensors.csv")

In [9]:
geo_names = set(sensors_geo["sensor_name"].str.strip())
trigger_names = set(triggers["sensor_name"].str.strip())

exact_matches = trigger_names & geo_names
unmatched = trigger_names - geo_names

print(f"{len(exact_matches)} names matched exactly")
print(f"{len(unmatched)} names did NOT match — needs review:")
for name in sorted(unmatched):
    print(f"  - {name}")

11 names matched exactly
0 names did NOT match — needs review:


In [10]:
merged = triggers.merge(
    sensors_geo,
    left_on="sensor_name",
    right_on="sensor_name",
    how="left"
)

# sanity check - should be zero
still_missing = merged[merged["latitude"].isna()]
print(f"{len(still_missing)} rows with no matched location")

display(merged.head())

0 rows with no matched location


,date,sensor_name,sensor_id,date_installed,street_name,borough,zipcode,census_tract,nta,latitude,...,tidally_influenced,date_removed,deployment_id,dev_id,sensor_status,date_updated,deploy_type,date_deployed,requested_by,in_floodplain
0,2025-01-12,SI - McLaughlin St/Agnes Pl,SI-mclaughlin-st-agnes-pl-1aigk0,2023-03-24T00:00:00.000,McLaughlin Street,Staten Island,10305,5007002,SI0201,40.589132,...,Yes,NaN,likely-well-thrush,fs5-00419,good,2026-06-03T20:11:48.826Z,coastal,2023-03-24T00:00:00,[],True
1,2025-01-14,SI - Grimsby St/ Mapleton Ave,SI-grimsby-st-mapleton-ave-1de5w0,2023-05-19T00:00:00.000,Grimsby Street,Staten Island,10306,5011203,SI0202,40.574836,...,Yes,NaN,purely_fancy_kite,fs5-00836,good,2026-06-03T15:16:25.849Z,coastal,2023-05-19T00:00:00,['dep'],True
2,2025-01-15,SI - Grimsby St/ Mapleton Ave,SI-grimsby-st-mapleton-ave-1de5w0,2023-05-19T00:00:00.000,Grimsby Street,Staten Island,10306,5011203,SI0202,40.574836,...,Yes,NaN,purely_fancy_kite,fs5-00836,good,2026-06-03T15:16:25.849Z,coastal,2023-05-19T00:00:00,['dep'],True
3,2025-01-16,Q - Beach Channel Dr/Beach 48th St,Q-beach-channel-dr-beach-48th-st-1zdub0,2024-07-19T00:00:00.000,Beach 48th Street,Queens,11691,4097204,QN1402,40.595001,...,Yes,NaN,really_usable_ghoul,fs3-00277,good,2026-01-09T18:26:00.687Z,coastal,2024-07-19T12:09:00,['community'],True
4,2025-01-18,SI - Grimsby St/ Mapleton Ave,SI-grimsby-st-mapleton-ave-1de5w0,2023-05-19T00:00:00.000,Grimsby Street,Staten Island,10306,5011203,SI0202,40.574836,...,Yes,NaN,purely_fancy_kite,fs5-00836,good,2026-06-03T15:16:25.849Z,coastal,2023-05-19T00:00:00,['dep'],True


In [11]:
import numpy as np

weather = df  # precipitation data loaded in the "Understanding the Data" section above

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

## Was it actually raining where the sensor is?

Rain in NYC is patchy, and some of these dates are mixed rain/snow days -- a
citywide "did any station report rain" flag would get confused by both. Instead,
for each (date, sensor) pair, look up what *that sensor's nearest station
specifically* recorded. This tells us whether the event a sensor picked up was
plausibly real local rain (a flood the public FloodNet dataset should have
captured, but didn't), versus a reading with no nearby rain to explain it.

In [12]:
station_locations = weather.drop_duplicates("station_id")[["station_id", "latitude", "longitude"]]

def nearest_station(lat, lon):
    dists = station_locations.apply(
        lambda row: haversine_km(lat, lon, row["latitude"], row["longitude"]), axis=1
    )
    idx = dists.idxmin()
    return station_locations.loc[idx, "station_id"], dists[idx]

sensor_stations = merged.drop_duplicates("sensor_name")[["sensor_name", "latitude", "longitude"]].copy()
sensor_stations[["nearest_station_id", "distance_km"]] = sensor_stations.apply(
    lambda row: pd.Series(nearest_station(row["latitude"], row["longitude"])), axis=1
)
print(sensor_stations)

                               sensor_name   latitude  longitude  \
0              SI - McLaughlin St/Agnes Pl  40.589132 -74.072692   
1            SI - Grimsby St/ Mapleton Ave  40.574836 -74.093448   
3       Q - Beach Channel Dr/Beach 48th St  40.595001 -73.779035   
5    Q - Beach 49th St/Rockaway Beach Blvd  40.593339 -73.779743   
8              SI - Baden Pl/ Mapleton Ave  40.573189 -74.090386   
30               BX - Watson Ave/Close Ave  40.825626 -73.882366   
39         SI - Hylan Blvd/ Jefferson Blvd  40.581063 -74.098494   
40                BX - Tier St/William Ave  40.848094 -73.789491   
48        SI - Snug Harbor Rd / Kissel Ave  40.644221 -74.106407   
55         SI - Minthorne St/ Victory Blvd  40.637624 -74.075250   
157            SI - Bedford Ave/Kiswick St  40.575318 -74.095266   

    nearest_station_id  distance_km  
0    GHCND:US1NYRC0002     5.614919  
1    GHCND:US1NYRC0002     3.416911  
3    GHCND:USW00094789     5.072638  
5    GHCND:USW00094789     5.26

In [13]:
import folium

station_coords = station_locations.set_index("station_id")[["latitude", "longitude"]]

m = folium.Map(location=[40.65, -73.95], zoom_start=10, tiles="cartodbpositron")

for _, row in sensor_stations.iterrows():
    station_lat, station_lon = station_coords.loc[row["nearest_station_id"]]

    folium.CircleMarker(
        [row["latitude"], row["longitude"]], radius=6, color="#2a78d6", fill=True,
        fill_opacity=1, tooltip=f"Sensor: {row['sensor_name']}",
    ).add_to(m)
    folium.CircleMarker(
        [station_lat, station_lon], radius=6, color="#eb6834", fill=True,
        fill_opacity=1, tooltip=f"Station: {row['nearest_station_id']}",
    ).add_to(m)
    folium.PolyLine(
        [[row["latitude"], row["longitude"]], [station_lat, station_lon]],
        color="#898781", weight=1, dash_array="4",
        tooltip=f"{row['distance_km']:.2f} km",
    ).add_to(m)

legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
            background: white; padding: 10px; border: 1px solid #c3c2b7; font-size: 13px;">
  <div><span style="color:#2a78d6;">&#9679;</span> Sensor</div>
  <div><span style="color:#eb6834;">&#9679;</span> Nearest weather station</div>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m


In [14]:
# Attach each (date, sensor) trigger to its sensor's nearest station,
# then look up what that specific station recorded on that exact date.
weather["date"] = pd.to_datetime(weather["date"])

hyperlocal = merged[["date", "sensor_name"]].merge(
    sensor_stations[["sensor_name", "nearest_station_id", "distance_km"]],
    on="sensor_name",
    how="left",
).merge(
    weather[["station_id", "date", "precip_type", "precipitation_in"]],
    left_on=["nearest_station_id", "date"],
    right_on=["station_id", "date"],
    how="left",
)

hyperlocal["actually_rained"] = hyperlocal["precip_type"] == "rain"

display(hyperlocal[[
    "date", "sensor_name", "nearest_station_id", "distance_km",
    "precip_type", "precipitation_in", "actually_rained",
]])

,date,sensor_name,nearest_station_id,distance_km,precip_type,precipitation_in,actually_rained
0,2025-01-12,SI - McLaughlin St/Agnes Pl,GHCND:US1NYRC0002,5.614919,none,0.0,False
1,2025-01-14,SI - Grimsby St/ Mapleton Ave,GHCND:US1NYRC0002,3.416911,none,0.0,False
2,2025-01-15,SI - Grimsby St/ Mapleton Ave,GHCND:US1NYRC0002,3.416911,none,0.0,False
3,2025-01-16,Q - Beach Channel Dr/Beach 48th St,GHCND:USW00094789,5.072638,none,0.0,False
4,2025-01-18,SI - Grimsby St/ Mapleton Ave,GHCND:US1NYRC0002,3.416911,NaN,NaN,False
...,...,...,...,...,...,...,...
222,2025-12-28,SI - Grimsby St/ Mapleton Ave,GHCND:US1NYRC0002,3.416911,NaN,NaN,False
223,2025-12-29,SI - Grimsby St/ Mapleton Ave,GHCND:US1NYRC0002,3.416911,NaN,NaN,False
224,2025-12-30,SI - Grimsby St/ Mapleton Ave,GHCND:US1NYRC0002,3.416911,NaN,NaN,False
225,2025-12-31,SI - Bedford Ave/Kiswick St,GHCND:US1NYRC0002,3.282136,NaN,NaN,False


In [15]:
# Summary: how many manually-flagged sensor events had real, local rain
# to explain them (candidates for "should be in the public dataset but isn't")
# vs. no nearby rain at all (a different question -- what caused those readings?)
n_total = len(hyperlocal)
n_rain = hyperlocal["actually_rained"].sum()
n_no_reading = hyperlocal["precip_type"].isna().sum()
n_other = n_total - n_rain - n_no_reading

print(f"{n_rain}/{n_total} sensor-triggered events had rain at their nearest station")
print(f"{n_other}/{n_total} had a nearby reading but it wasn't rain (snow / none)")
print(f"{n_no_reading}/{n_total} had no precip reading for that date at the nearest station")

37/227 sensor-triggered events had rain at their nearest station
111/227 had a nearby reading but it wasn't rain (snow / none)
79/227 had no precip reading for that date at the nearest station


In [16]:
# see which sensor-triggered events had rain
display(hyperlocal[hyperlocal["actually_rained"]][["date", "sensor_name", "nearest_station_id", "distance_km", "precip_type", "precipitation_in"]])
display(sorted(hyperlocal.loc[hyperlocal["actually_rained"], "date"].dt.strftime("%Y-%m-%d").unique()))

,date,sensor_name,nearest_station_id,distance_km,precip_type,precipitation_in
20,2025-01-31,Q - Beach Channel Dr/Beach 48th St,GHCND:USW00094789,5.072638,rain,0.015
33,2025-02-13,BX - Watson Ave/Close Ave,GHCND:USW00014732,5.137567,rain,0.018
34,2025-02-16,BX - Watson Ave/Close Ave,GHCND:USW00014732,5.137567,rain,0.079
44,2025-03-06,SI - Baden Pl/ Mapleton Ave,GHCND:US1NYRC0002,3.631422,rain,0.102
58,2025-03-16,BX - Tier St/William Ave,GHCND:US1NYNS0027,9.242663,rain,0.002
62,2025-03-21,SI - Hylan Blvd/ Jefferson Blvd,GHCND:US1NYRC0002,3.264544,rain,0.113
63,2025-03-21,BX - Tier St/William Ave,GHCND:US1NYNS0027,9.242663,rain,0.039
65,2025-03-24,SI - McLaughlin St/Agnes Pl,GHCND:US1NYRC0002,5.614919,rain,0.014
69,2025-03-27,SI - McLaughlin St/Agnes Pl,GHCND:US1NYRC0002,5.614919,rain,0.004
72,2025-03-30,BX - Watson Ave/Close Ave,GHCND:USW00014732,5.137567,rain,0.005


['2025-01-31',
 '2025-02-13',
 '2025-02-16',
 '2025-03-06',
 '2025-03-16',
 '2025-03-21',
 '2025-03-24',
 '2025-03-27',
 '2025-03-30',
 '2025-04-03',
 '2025-04-04',
 '2025-04-05',
 '2025-04-13',
 '2025-04-26',
 '2025-05-04',
 '2025-05-06',
 '2025-06-10',
 '2025-06-23',
 '2025-07-25',
 '2025-07-31',
 '2025-09-05',
 '2025-09-18',
 '2025-10-30',
 '2025-11-08',
 '2025-11-10',
 '2025-11-26',
 '2025-12-02',
 '2025-12-19']

In [17]:
display(hyperlocal['sensor_name'].unique())

<StringArray>
[          'SI - McLaughlin St/Agnes Pl',
         'SI - Grimsby St/ Mapleton Ave',
    'Q - Beach Channel Dr/Beach 48th St',
 'Q - Beach 49th St/Rockaway Beach Blvd',
           'SI - Baden Pl/ Mapleton Ave',
             'BX - Watson Ave/Close Ave',
       'SI - Hylan Blvd/ Jefferson Blvd',
              'BX - Tier St/William Ave',
      'SI - Snug Harbor Rd / Kissel Ave',
       'SI - Minthorne St/ Victory Blvd',
           'SI - Bedford Ave/Kiswick St']
Length: 11, dtype: str

# From 17 mismatched sensors to 11 verified-gap sensors

## How this whole analysis fits together
1. **Tidal classification** (`2_tidal_analysis/2_tidal_analysis.ipynb`) — a Rayleigh/lag
   test labeled each sensor tidal vs. non-tidal from its flood-event timing.
2. **Mismatches** — sensors where the test disagreed with the existing
   `tidally_influenced` label. 27 total; 10 are "Insufficient data", leaving
   **17 with sufficient data = 12 false negatives + 5 false positives**.
3. **Manual review of the 17 → spotted a 2025 gap.** Went looking for a
   *classification* problem, found a *data* problem.
4. **Sized the gap** (`FloodNet_2025_Data_Gap.ipynb`) — network-wide: 111 of 415
   sensors silent all of 2025; API truncation, retirement, and hardware ruled out.
5. **Loop back, event by event** (`FloodNet_Data_Manual_QC.csv`) — for each sensor,
   logged every 2025 depth bump on the floodnet.nyc graph that is missing from the
   public feed. **11 of the 17** had such gaps to log.
6. **Weather corroboration** (this notebook) — hyperlocal, nearest-station rain
   check on each logged event.

## Why 17 → 11
The 6 that dropped (verified in the next cell) are **3 false positives**
(Brighton 6th St, Q-101st St/160th Ave, Beach Chn Dr/Beach 59th St) and **3 false
negatives** (Brookville Blvd/149th St, Norton Dr/Westbourne Ave, Boundary Ave/Hamden
Ave). They dropped because the manual graph review found **nothing to log** — the
graph and the feed agreed, so there was no missing event — not because the analysis
broke. Keeping only the 11 with *positive graph evidence* of a missing flood (and
dropping the ones where graph and feed agree) is deliberate curation, not an
oversight.

## Where "is it really missing?" verification stands
- **Already have (sensor-level):** `2_tidal_analysis.ipynb`'s "Verifying the silence
  against the source directly" fetches raw events straight from `aq7i-eu5q`,
  bypassing every join, and confirms a fully-silent sensor has *zero* 2025 events —
  by name.
- **Baked into the QC table:** absence-from-feed is *how the QC list was built* —
  every row is an event seen on the graph and not found in the feed. So an
  event-level programmatic check is a **reproducible cross-check against manual
  slips** (an event mis-read as missing that's actually in the feed a day off in
  UTC), not new information.
- **Now automated (rain-anchored, see "Genuinely missing, or just delayed?" below):**
  an event-level, `sensor_id`-matched, ±1-day feed check of the *specific* QC dates.
  Of the **63 rain-driven QC events, 62 are genuinely missing** from the feed, **1 is
  a QC slip** (SI - Grimsby St/Mapleton Ave on 2025-12-19 is actually in the feed
  same-day), and **0 were merely delayed** — widening the window forward rescued
  nothing, so the gaps are real absences, not date offsets. The remaining 164/227 QC
  events have no rain driver (tidal floods rain can't adjudicate) and are left to the
  tidal-classification pass — this rain-anchored check deliberately doesn't opine on
  them.

In [18]:
# Verify why the 6 mismatched sensors dropped out of the QC list: check whether the
# public feed already has (or lacks) their 2025 events. If a dropped sensor has zero
# 2025 events AND its graph showed no 2025 floods, there was nothing to log; if the
# feed already carries its events, graph and feed agree -- either way, no gap.
from urllib.parse import urlencode, quote

def soda_url(resource, **params):
    return f'https://data.cityofnewyork.us/resource/{resource}.json?' + urlencode(params, quote_via=quote)

def normalize_name(n):
    return ' '.join(str(n).strip().lower().split())

dropped_sensors = [
    'BK - Brighton 6th St/Ocean View Ave', 'Q - 101st St/160th Ave', 'Q - Beach Chn Dr/ Beach 59th St',
    'Q - Brookville Blvd/ 149th St', 'Q - Norton Dr/ Westbourne Ave', 'SI - Boundary Ave/Hamden Ave',
]
sid_map = {normalize_name(r.sensor_name): r.sensor_id
           for r in sensors_geo[['sensor_name', 'sensor_id']].drop_duplicates().itertuples()}

# Full event history, verified against count(*) so a silent row can't be a truncated fetch.
feed = pd.read_json(soda_url('aq7i-eu5q', **{'$select': 'sensor_id, sensor_name, flood_start_time', '$limit': 50000}))
feed_count = int(pd.read_json(soda_url('aq7i-eu5q', **{'$select': 'count(*)'}))['count'][0])
assert len(feed) == feed_count, f"truncated: {len(feed)} vs {feed_count}"
feed['_key'] = feed['sensor_name'].apply(normalize_name)
feed['yr'] = pd.to_datetime(feed['flood_start_time']).dt.year

print(f"feed verified untruncated: {len(feed)} events\n")
for name in dropped_sensors:
    key = normalize_name(name)
    sub = feed[(feed['_key'] == key) | (feed['sensor_id'] == sid_map.get(key))]
    by_yr = sub['yr'].value_counts().sort_index().to_dict()
    print(f"{name:40s} | 2025 events: {int((sub['yr'] == 2025).sum()):3d} | all years: {by_yr}")

feed verified untruncated: 2604 events

BK - Brighton 6th St/Ocean View Ave      | 2025 events:   0 | all years: {2026: 5}
Q - 101st St/160th Ave                   | 2025 events:   0 | all years: {2026: 3}
Q - Beach Chn Dr/ Beach 59th St          | 2025 events:   0 | all years: {2026: 6}
Q - Brookville Blvd/ 149th St            | 2025 events:   2 | all years: {2025: 2, 2026: 2}
Q - Norton Dr/ Westbourne Ave            | 2025 events:   0 | all years: {2026: 7}
SI - Boundary Ave/Hamden Ave             | 2025 events:   0 | all years: {2026: 8}


**Result:** 5 of the 6 dropped sensors have **zero** 2025 events in the feed and
only start appearing in 2026 — and their deploy dates confirm why: **all 5 were
deployed in 2026**, so there was no hardware in the ground in 2025 to record
anything. Zero 2025 events is expected, not a gap. Brookville Blvd/149th St has 2
events in 2025, both already in the feed. None shows a graph-vs-feed discrepancy for
2025 — consistent with the sensors not existing yet (the five deployed in 2026) or
the feed already carrying the events (Brookville). This is why they were correctly
excluded: the QC list holds only sensors with positive graph evidence of a flood the
feed is missing, not every quiet sensor. (The feed side is confirmed here; the graph
side is the manual review, and the two are consistent.)

# Genuinely missing, or just delayed?

Everything above asks flood -> rain: was each suspected-missing flood
meteorologically plausible? This flips it to rain -> flood to separate two very
different explanations for a graph event the public feed lacks:

- **genuinely missing** — the feed has no flood for that sensor on the flood day or
  the day after, or
- **just delayed** — the feed *does* carry it, a day off (UTC boundary / processing
  lag), and it was hand-flagged as missing.

We gate on a rain driver (rain at the nearest station on the flood day or the day
before), then look forward into the feed. Non-rain (likely tidal) events fall out by
design — rain can't anchor them, and they belong to the separate tidal-classification
question. This is the event-level, `sensor_id`-matched ±1-day feed check flagged as
"not yet automated" in the section above.

In [19]:
# ---------------------------------------------------------------------------
# Rain-anchored, event-level check: genuinely missing vs. just delayed.
#
# The checks above go flood -> rain (was each suspected-missing flood plausible?).
# Flip it to rain -> flood. For each QC-logged event we (1) require a rain driver --
# rain at the nearest station on the flood day OR the day before -- then (2) look
# FORWARD into the public feed: does the feed carry a flood for that sensor on the
# flood day (on time) or the next day (delayed 1d)? If neither -> genuinely missing.
# This catches the failure mode where an event really is in the feed, just a day off
# (UTC boundary / processing lag), and was hand-flagged as missing.
#
# Non-rain (likely tidal) QC events fall out here by design: rain can't anchor them,
# and they belong to the separate tidal-classification question. Reuses feed / _key /
# normalize_name / sid_map from the "Verify why the 6 dropped" cell above.
# ---------------------------------------------------------------------------

# (1) full-year daily rain type for just the nearest stations (~5 unique)
station_daily = {}
for sid in sensor_stations["nearest_station_id"].unique():
    for date_str, d in get_station_precip(sid).items():
        if d.get("PRCP") is None:
            continue
        tmin = d.get("TMIN")
        station_daily[(sid, date_str)] = classify_precip(
            d["PRCP"] / 10.0, d.get("SNOW"), tmin / 10.0 if tmin is not None else None
        )

def has_rain_driver(station_id, day):
    # flood follows rain: rain the same day or the day before
    return any(
        station_daily.get((station_id, (day + pd.Timedelta(days=k)).strftime("%Y-%m-%d"))) == "rain"
        for k in (-1, 0)
    )

# (2) feed flood dates per sensor (all years, so a delayed event at a year boundary
#     still matches), keyed for lookup by exact date
feed["fdate"] = pd.to_datetime(feed["flood_start_time"]).dt.normalize()

def feed_flood_dates(sensor_name, sensor_id):
    key = normalize_name(sensor_name)
    sub = feed[(feed["_key"] == key) | (feed["sensor_id"] == sensor_id)]
    return set(sub["fdate"])

rows = []
for r in hyperlocal.itertuples():
    D = r.date
    if not has_rain_driver(r.nearest_station_id, D):
        status = "no rain driver"
    else:
        fd = feed_flood_dates(r.sensor_name, sid_map.get(normalize_name(r.sensor_name)))
        if D in fd:
            status = "in feed (on time)"
        elif (D + pd.Timedelta(days=1)) in fd:
            status = "in feed (delayed 1d)"
        else:
            status = "genuinely missing"
    rows.append({"date": D, "sensor_name": r.sensor_name,
                 "nearest_station_id": r.nearest_station_id, "status": status})

missing_check = pd.DataFrame(rows)

print(missing_check["status"].value_counts().to_string())
rain_assoc = missing_check[missing_check["status"] != "no rain driver"]
print(f"\n{len(rain_assoc)}/{len(missing_check)} QC events had a rain driver "
      f"(the rest are non-rain / likely tidal, handled separately)")

# The actionable rows: events the feed actually carries (on time or a day late) --
# i.e. hand-flagged as missing but not genuinely absent.
in_feed = missing_check[missing_check["status"].str.startswith("in feed")]
print(f"{len(in_feed)} rain-driven QC events are actually IN the feed (QC over-flags to correct)")
display(in_feed.sort_values(["sensor_name", "date"]))

status
no rain driver       164
genuinely missing     62
in feed (on time)      1

63/227 QC events had a rain driver (the rest are non-rain / likely tidal, handled separately)
1 rain-driven QC events are actually IN the feed (QC over-flags to correct)


,date,sensor_name,nearest_station_id,status
213,2025-12-19,SI - Grimsby St/ Mapleton Ave,GHCND:US1NYRC0002,in feed (on time)


In [20]:
# Export the per-event rain-driver check for reuse in Tidal_Corroboration.ipynb, which
# needs this automated split instead of the manual QC "Weather" column flag.
missing_check.to_csv("qc_rain_driver_check.csv", index=False)
print(f"wrote qc_rain_driver_check.csv ({len(missing_check)} rows)")
missing_check["status"].value_counts()

wrote qc_rain_driver_check.csv (227 rows)


status
no rain driver       164
genuinely missing     62
in feed (on time)      1
Name: count, dtype: int64

**Result:** of the 63 rain-driven QC events, **62 are genuinely missing** from the
public feed and **0 were merely delayed** — looking forward a day rescued nothing, so
these are real absences, not date-offset artifacts. The remaining 164/227 QC events
have no rain driver (tidal floods) and aren't adjudicated here.

> ⚠️ **One QC row to re-check manually:** `SI - Grimsby St/Mapleton Ave` on
> **2025-12-19** came back **"in feed (on time)"** — it's actually present in
> `aq7i-eu5q` that same day, so it should *not* have been logged as missing. Flagging
> here only; **not** editing `FloodNet_Data_Manual_QC.csv`. It doesn't cost Grimsby
> its gap-sensor status (it has other genuinely-missing dates), just this one row.